In [1]:
import os
import torch
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [16]:
def calculate_interval_score(y_true, lower, upper, alpha):
    """Gneiting & Raftery Interval Score (Lower is better)"""
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2.0 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2.0 / alpha) * (y_true - upper), 0)
    return np.mean(width + penalty_lower + penalty_upper)

def run_cqr_baseline(split_dir="split_embeddings"):
    print("Starting Conformalized Quantile Regression (CQR) Baseline...\n")
    category_t_map = {
        "BIKES": 0.94, "BOOKS": 0.97, "CARS": 0.97, "CYCLE": 0.97,
        "FLAT": 0.96, "FRIDGES": 0.97, "GAMES": 0.97, "GAMESENTERTAINMENT": 0.96,
        "LAPTOP": 0.97, "MOBILE": 0.95, "PHONES": 0.96, "PRINTER": 0.95,
        "TV": 0.97, "WASHINGMACHINE": 0.94
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name, 0.95)
        alpha = 1.0 - t_val
        
        print(f"\nCQR BASELINE: {cat_name}\n{'-'*60}")
        
        # 1. Load Data
        raw_train_data = torch.load(train_path, weights_only=False)
        raw_test_data = torch.load(test_path, weights_only=False)
        
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        # Convert to numpy arrays for LightGBM
        X_full_train = raw_train_data['embeddings'].numpy()
        y_full_train = ((raw_train_data['log_prices'] - cat_mu) / cat_sigma).numpy()
        
        X_test = raw_test_data['embeddings'].numpy()
        y_test_real = np.exp(raw_test_data['log_prices'].numpy()) # Real rupees
        
        # Handle tiny datasets (like GAMES) where split conformal might fail
        if len(X_full_train) < 30:
            print("   -> Dataset too small for proper Conformal Calibration. Skipping.")
            continue
            
        # 2. Split into Train (80%) and Calibration (20%)
        X_train, X_calib, y_train, y_calib = train_test_split(
            X_full_train, y_full_train, test_size=0.2, random_state=54
        )
        
        # 3. Train Base Quantile Models
        lgb_lower = lgb.LGBMRegressor(objective='quantile', alpha=alpha/2, n_estimators=100, random_state=42, verbose=-1)
        lgb_upper = lgb.LGBMRegressor(objective='quantile', alpha=1-(alpha/2), n_estimators=100, random_state=42, verbose=-1)
        
        lgb_lower.fit(X_train, y_train)
        lgb_upper.fit(X_train, y_train)
        
        # 4. Conformal Calibration Phase
        # Predict on calibration set
        calib_preds_lower = lgb_lower.predict(X_calib)
        calib_preds_upper = lgb_upper.predict(X_calib)
        
        # Calculate Conformity Scores: How far outside the bounds did the true price fall?
        error_lower = calib_preds_lower - y_calib
        error_upper = y_calib - calib_preds_upper
        conformity_scores = np.maximum(error_lower, error_upper)
        
        # Calculate the empirical quantile (the adjustment factor 'q')
        n = len(y_calib)
        q_level = np.ceil((n + 1) * t_val) / n
        q_level = min(max(q_level, 0.0), 1.0) # Ensure it stays between 0 and 1
        q_hat = np.quantile(conformity_scores, q_level)
        
        # 5. Testing Phase
        test_preds_lower = lgb_lower.predict(X_test)
        test_preds_upper = lgb_upper.predict(X_test)
        
        # Apply the Conformal Adjustment to guarantee coverage
        adjusted_lower = test_preds_lower - q_hat
        adjusted_upper = test_preds_upper + q_hat
        
        # Inverse Transform bounds back to Real Rupees
        l_real = np.exp(adjusted_lower * cat_sigma + cat_mu)
        u_real = np.exp(adjusted_upper * cat_sigma + cat_mu)
        
        # 6. Calculate Metrics
        picp = np.mean((y_test_real >= l_real) & (y_test_real <= u_real))
        mpiw = np.mean(u_real - l_real)
        mid_preds = (l_real + u_real) / 2.0
        rmse = np.sqrt(np.mean((y_test_real - mid_preds) ** 2))
        interval_score = calculate_interval_score(y_test_real, l_real, u_real, alpha=alpha)
        
        print(f"-> Calibration Factor (q): {q_hat:.4f}")
        print(f"-> CQR PICP (Coverage):    {picp:.4f}")
        print(f"-> CQR MPIW (Width):       ₹{mpiw:,.2f}")
        print(f"-> CQR RMSE (Midpoint):    ₹{rmse:,.2f}")
        print(f"-> CQR Interval Score:     {interval_score:,.2f}")



In [13]:
torch.cuda.empty_cache()
run_cqr_baseline()

Starting Conformalized Quantile Regression (CQR) Baseline...


CQR BASELINE: BIKES
------------------------------------------------------------
-> Calibration Factor (q): 1.3384
-> CQR PICP (Coverage):    1.0000
-> CQR MPIW (Width):       ₹1,152,644.62
-> CQR RMSE (Midpoint):    ₹486,338.57
-> CQR Interval Score:     1,152,644.62

CQR BASELINE: BOOKS
------------------------------------------------------------
-> Calibration Factor (q): 0.8082
-> CQR PICP (Coverage):    0.9783
-> CQR MPIW (Width):       ₹17,207.26
-> CQR RMSE (Midpoint):    ₹7,415.40
-> CQR Interval Score:     18,060.75

CQR BASELINE: CARS
------------------------------------------------------------
-> Calibration Factor (q): 0.2257
-> CQR PICP (Coverage):    0.8710
-> CQR MPIW (Width):       ₹7,184,075.47
-> CQR RMSE (Midpoint):    ₹3,247,732.45
-> CQR Interval Score:     7,381,426.66

CQR BASELINE: CYCLE
------------------------------------------------------------
-> Calibration Factor (q): 0.4001
-> CQR PICP (Covera

In [17]:
torch.cuda.empty_cache()
run_cqr_baseline()

Starting Conformalized Quantile Regression (CQR) Baseline...


CQR BASELINE: BIKES
------------------------------------------------------------
-> Calibration Factor (q): 1.4018
-> CQR PICP (Coverage):    1.0000
-> CQR MPIW (Width):       ₹1,169,381.41
-> CQR RMSE (Midpoint):    ₹493,416.24
-> CQR Interval Score:     1,169,381.41

CQR BASELINE: BOOKS
------------------------------------------------------------
-> Calibration Factor (q): 0.8849
-> CQR PICP (Coverage):    1.0000
-> CQR MPIW (Width):       ₹17,480.85
-> CQR RMSE (Midpoint):    ₹7,213.60
-> CQR Interval Score:     17,480.85

CQR BASELINE: CARS
------------------------------------------------------------
-> Calibration Factor (q): 0.2257
-> CQR PICP (Coverage):    0.8710
-> CQR MPIW (Width):       ₹7,184,075.47
-> CQR RMSE (Midpoint):    ₹3,247,732.45
-> CQR Interval Score:     7,381,426.66

CQR BASELINE: CYCLE
------------------------------------------------------------
-> Calibration Factor (q): 0.2108
-> CQR PICP (Covera